# NeMo AutoModel on SageMaker — Training Job の起動

既存の Composer 版 Notebook (`HuggingFace` Estimator + `launcher_sft.py`) の置き換えです。

| 項目 | Composer 版 | 本 Notebook |
|---|---|---|
| Estimator | `HuggingFace(image_uri=..., entry_point='launcher_sft.py')` | `PyTorch(image_uri=..., entry_point='train.py')` |
| 分散起動 | `launcher_sft.py` が `composer -n 8` を subprocess 起動 | `distribution={'torch_distributed': {'enabled': True}}` で toolkit が torchrun 起動 |
| モデル | rank0 が boto3 で `model.tar.gz` を DL | HF Hub から直接、または `model` チャネル |
| データ | `train` チャネルの JSON | `train` / `validation` チャネルの JSONL |
| 設定 | fire の引数 | `configs/sagemaker/*.yaml` + hyperparameters で上書き |


In [ ]:
import os, json, time, tarfile, boto3, sagemaker
from sagemaker.pytorch import PyTorch

sess    = sagemaker.Session()
# SageMaker Studio / Notebook Instance 上ではロールを自動取得。ローカル Jupyter では環境変数 SAGEMAKER_ROLE
# (arn:aws:iam::<account>:role/<SageMaker 実行ロール>) を指定する。
try:
    role = os.environ.get('SAGEMAKER_ROLE') or sagemaker.get_execution_role()
except Exception:
    raise SystemExit('ローカル実行時は SAGEMAKER_ROLE 環境変数に SageMaker 実行ロールの ARN を設定してください')
region  = boto3.Session().region_name
account = boto3.client('sts').get_caller_identity()['Account']
bucket  = sess.default_bucket()          # 既存のデータバケットを使うなら書き換える

IMAGE_REPO = 'nemo-automodel-sagemaker'
IMAGE_TAG  = '0.6.0-pt2.10-py313-cu130'   # container/build_and_push.sh が push したタグ
image_uri  = f'{account}.dkr.ecr.{region}.amazonaws.com/{IMAGE_REPO}:{IMAGE_TAG}'

print('sagemaker', sagemaker.__version__, '| region', region, '| bucket', bucket)
print('image_uri', image_uri)


## 実行構成（ここだけ変えれば以降のセルが連動する）

`target` でインスタンスを選ぶと、GPU 数・クォータ名・`global_batch_size`・チェックポイントの prefix（`RUN_TAG`）が連動します。


In [ ]:
INSTANCES = {
    #          instance_type          GPU数  1GPUあたりの pack 数 (local_batch_size)
    'cheap': dict(instance_type='ml.g5.2xlarge',    gpus=1, local_batch=1),   # 1×A10G 24GB。最初の疎通確認向け
    'multi': dict(instance_type='ml.g5.12xlarge',   gpus=4, local_batch=1),   # 4×A10G 24GB。FSDP2 の複数 GPU 確認 (安価)
    'p4d':   dict(instance_type='ml.p4d.24xlarge',  gpus=8, local_batch=1),   # 8×A100 40GB。8 GPU の動作確認
    'prod':  dict(instance_type='ml.p4de.24xlarge', gpus=8, local_batch=2),   # 8×A100 80GB。7B〜13B 級
}
target         = 'cheap'
instance_count = 1
grad_accum     = 1      # 勾配蓄積回数。global_batch_size = local_batch × gpus × instance_count × grad_accum (単位は pack)

inst              = INSTANCES[target]
instance_type     = inst['instance_type']
world_size        = inst['gpus'] * instance_count
local_batch_size  = inst['local_batch']
global_batch_size = local_batch_size * world_size * grad_accum

# チェックポイントの S3 prefix。AutoModel は同じ prefix の最新チェックポイントから自動再開するため (Spot 中断からの復帰に使う)、
# target やモデル・PEFT 設定を変えたら prefix も変わるように target を含める。同じ構成でやり直すときは末尾の v1 を上げるか fresh_start=1。
RUN_TAG = f'qwen35-0.8b-lora-mtp-{target}-v1'

print(f'target={target} | {instance_type} x{instance_count} | world_size={world_size} | local_batch={local_batch_size} | global_batch={global_batch_size} pack')
print('RUN_TAG', RUN_TAG)


## 0. プリフライト確認

ジョブを投げる前に、(a) ECR にイメージが同一リージョンにあること、(b) 対象インスタンスの Training Job クォータがあること、
(c) 実行ロールが ECR / S3 にアクセスできることを確認します。ここで失敗するものは `fit()` でも失敗します。


In [ ]:
# (a) ECR イメージ (Training Job と同一リージョンにあること)
ecr = boto3.client('ecr', region_name=region)
imgs = ecr.describe_images(repositoryName=IMAGE_REPO, imageIds=[{'imageTag': IMAGE_TAG}])['imageDetails']
print('ECR image :', IMAGE_TAG, f"{imgs[0]['imageSizeInBytes']/1e9:.1f} GB (compressed), pushed", imgs[0]['imagePushedAt'])

# (b) Training Job のインスタンスクォータ (0 なら Service Quotas で引き上げ申請が必要)
#     Studio の実行ロールに servicequotas:ListServiceQuotas が無いことが多いので、その場合はコンソールで目視確認する
quota_name = f'{instance_type} for training job usage'   # 上の実行構成セルの target に連動
try:
    sq = boto3.client('service-quotas', region_name=region)
    pages = sq.get_paginator('list_service_quotas').paginate(ServiceCode='sagemaker')
    quotas = [q for pg in pages for q in pg['Quotas'] if q['QuotaName'] == quota_name]
    for q in quotas:
        print('quota     :', q['QuotaName'], '=', q['Value'], ('  <-- 0 のため申請が必要' if q['Value'] < 1 else ''))
    if not quotas:
        print('quota     : 見つからず。コンソールの Service Quotas > Amazon SageMaker で確認してください')
except Exception as e:
    print(f'quota     : 参照権限なし ({type(e).__name__})。コンソールの Service Quotas > AWS services > Amazon SageMaker で')
    print(f'            "{quota_name}" を検索し、Applied quota value が 1 以上であることを確認してください')

# (c) 実行ロールの確認
print('role      :', role)
print('bucket    :', bucket, '| region', region)


## 1. 学習データを S3 へ

`data/cooking_basics/{train,val}.jsonl` を `train` / `validation` チャネルとして渡します。
`output` は必ず文字列、`prompt` は `instruction + "\n\n" + input` の結合済みフィールドです (`data/README.md`)。


In [ ]:
prefix = 'automodel/cooking_basics'
train_s3 = sess.upload_data(path='../data/cooking_basics/train.jsonl', bucket=bucket, key_prefix=f'{prefix}/train')
val_s3   = sess.upload_data(path='../data/cooking_basics/val.jsonl',   bucket=bucket, key_prefix=f'{prefix}/validation')
print(train_s3); print(val_s3)


## 2. ベースモデルの渡し方

- **HF Hub から直接** (既定): `--model_id Qwen/Qwen3.5-0.8B`。ジョブがインターネットに出られる必要があります (VPC / network isolation なし)
- **`model` チャネル**: S3 上の非圧縮ディレクトリ (推奨) または `model.tar.gz` を `model_s3` に指定。
  `train.py` が `/opt/ml/input/data/model` を解決します (tar.gz は rank ごとに 1 回だけ展開)


In [ ]:
model_id = 'Qwen/Qwen3.5-0.8B'
model_s3 = None   # 例: f's3://{bucket}/pretrained_model/Qwen3.5-0.8B/'  (非圧縮 prefix)  or  '.../model.tar.gz'


## 3. Estimator

`hyperparameters` は toolkit が `--key value` に変換して `train.py` に渡します。
`train.py` は `config` / `set` / `model_id` / `warmup_epochs` / `rslora_alpha` / `final_checkpoint` を解釈し、
**それ以外の `a.b.c` 形式のキーはそのまま AutoModel の設定上書き**になります。


In [ ]:
hyperparameters = {
    'config': 'qwen3_5_cooking_lora.yaml',
    'model_id': model_id,
    'warmup_epochs': 1,
    'final_checkpoint': 'LOWEST_VAL',
    'fresh_start': 0,                  # 1 で既存チェックポイントを捨てて最初から
    # 以下は AutoModel の設定を直接上書き
    'step_scheduler.num_epochs': 3,
    # バッチは実行構成セルで GPU 数から導出 (単位は pack = 2048 トークン。cooking データは 1 エポック ≈ 17 pack。1 pack ≈ 14.5 GiB)
    'step_scheduler.global_batch_size': global_batch_size,
    'step_scheduler.local_batch_size': local_batch_size,
    'optimizer.lr': 1e-4,
    # 'model.attn_implementation': 'flash_attention_2',   # A10G/A100 以上で有効化を検討
}

metric_definitions = [
    {'Name': 'train:loss',      'Regex': r'step \d+ \| epoch \d+ \| loss ([0-9\.]+)'},
    {'Name': 'train:grad_norm', 'Regex': r'\| grad_norm ([0-9\.]+)'},
    {'Name': 'train:lr',        'Regex': r'\| lr ([0-9\.eE\+\-]+)'},
    {'Name': 'train:tps',       'Regex': r'\| tps ([0-9\.]+)'},
    {'Name': 'eval:loss',       'Regex': r'\[val\][^|]*\| step \d+ \| epoch \d+ \| loss ([0-9\.]+)'},
]

estimator = PyTorch(
    image_uri=image_uri,
    entry_point='train.py',
    source_dir='../src',                 # train.py
    dependencies=['../configs'],         # /opt/ml/code/configs/ に配置される
    role=role,
    base_job_name='automodel-qwen35-cooking-lora',
    instance_type=instance_type,
    instance_count=instance_count,
    volume_size=100,
    max_run=6*3600,
    distribution={'torch_distributed': {'enabled': True}},   # toolkit が torchrun を起動
    hyperparameters=hyperparameters,
    metric_definitions=metric_definitions,
    environment={
        'HF_HOME': '/tmp/hf',
        'HF_TOKEN': os.environ.get('HF_TOKEN', ''),           # gated モデルのときだけ
        'NCCL_DEBUG': 'WARN',
        'WANDB_MODE': 'disabled',
    },
    checkpoint_s3_uri=f's3://{bucket}/{prefix}/checkpoints/{RUN_TAG}/',   # /opt/ml/checkpoints と双方向同期 (自動再開)
    # keep_alive_period_in_seconds=1800,   # ウォームプール (クォータが必要)。反復時は有効化
)


## 4. 実行

初回は `wait=True` でログを流し、`=== effective config ===` と `=== Sample Prompt 0 ===` が意図通りか確認します。


In [ ]:
inputs = {'train': train_s3, 'validation': val_s3}
if model_s3:
    inputs['model'] = model_s3

print(f'{instance_type} x{instance_count} | global_batch={global_batch_size} | RUN_TAG={RUN_TAG}')
estimator.fit(inputs, wait=True, logs='All')
job_name = estimator.latest_training_job.name
print('job:', job_name)


## 5. 成果物の確認

`/opt/ml/model` の内容が `model.tar.gz` になります。中身は `model/adapter_model.safetensors` (LoRA アダプタ)、
`config.yaml` / `effective_config.yaml`、`tokenizer/`、`training.jsonl` / `validation.jsonl`、`training_info.json` です。


In [ ]:
desc = sess.sagemaker_client.describe_training_job(TrainingJobName=job_name)
model_data = desc['ModelArtifacts']['S3ModelArtifacts']
print('status :', desc['TrainingJobStatus'], '| billable sec:', desc.get('BillableTimeInSeconds'))
print('model  :', model_data)

os.makedirs('artifacts', exist_ok=True)
local_tar = f'artifacts/{job_name}.tar.gz'
sess.download_data(path='artifacts', bucket=model_data.split('/')[2], key_prefix='/'.join(model_data.split('/')[3:]))
with tarfile.open(os.path.join('artifacts', os.path.basename(model_data))) as t:
    t.extractall(f'artifacts/{job_name}')
    print('\n'.join(sorted(m.name for m in t.getmembers())))
print(open(f'artifacts/{job_name}/training_info.json').read())


In [ ]:
# 学習曲線 (training.jsonl / validation.jsonl は 1 行 1 ステップの JSON)
import pandas as pd
tr = pd.read_json(f'artifacts/{job_name}/training.jsonl', lines=True)
va = pd.read_json(f'artifacts/{job_name}/validation.jsonl', lines=True)
display(tr.tail()); display(va)
